# Privileged formation-order oracle with a matched control

This bounded diagnostic trains exactly two identical order-conditioned NAFNet models from the same official SIDD32 initialization. One receives the true generator order A/B; the other always receives a fixed zero code. The conditioner is zero-initialized, and every setting except the input code is shared.

The locked per-sample results from Fixed-A, Fixed-B, and Balanced-order are stored as a small JSON file in the repository, so no previous ZIP or checkpoint upload is required. Expected runtime is roughly 125–145 minutes on Kaggle 2xT4.

In [ ]:
import json
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/HoangKhanhTung0111/CoT-restoration.git"
REPO_DIR = Path("/kaggle/working/CoT-restoration")
if REPO_DIR.is_dir():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("Commit:", COMMIT)

In [ ]:
CDD11_ROOT = Path("/kaggle/input/datasets/mintesnotfikir/cdd-11-30")
PRETRAINED_ROOT = Path("/kaggle/input/datasets/hoangkhanhtung/nafnetmodel")
CONFIG = Path("configs/order_condition_oracle_low_haze.json")
REFERENCE = Path("configs/order_controls_reference_20260920.json")
EXPERIMENTS_ROOT = Path("/kaggle/working/experiments_order_condition_oracle")
EVALUATION_DIR = Path("/kaggle/working/order_condition_oracle_evaluation")

assert CDD11_ROOT.is_dir(), f"Missing CDD-11 input: {CDD11_ROOT}"
assert PRETRAINED_ROOT.is_dir(), f"Missing pretrained input: {PRETRAINED_ROOT}"
assert CONFIG.is_file() and REFERENCE.is_file()
gpu_names = subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True).strip().splitlines()
assert len(gpu_names) == 2 and all("T4" in name for name in gpu_names), f"Select Kaggle 2xT4; found {gpu_names}"
print("GPUs:", gpu_names)

In [ ]:
subprocess.run([
    "python", "-m", "hybrid_cot_nafnet.audit_kaggle",
    "--data-root", str(CDD11_ROOT),
    "--pretrained-root", str(PRETRAINED_ROOT),
    "--output", "/kaggle/working/order_condition_input_audit.json",
], check=True)
subprocess.run([
    "python", "-m", "hybrid_cot_nafnet.audit_order_control_data",
    "--data-root", str(CDD11_ROOT),
    "--output", "/kaggle/working/order_condition_data_gate.json",
], check=True)
subprocess.run([
    "python", "-m", "hybrid_cot_nafnet.run_ablation",
    "--config", str(CONFIG),
    "--data-root", str(CDD11_ROOT),
    "--experiments-root", str(EXPERIMENTS_ROOT),
    "--nproc-per-node", "2",
    "--dry-run",
], check=True)

In [ ]:
subprocess.run([
    "python", "-m", "hybrid_cot_nafnet.run_ablation",
    "--config", str(CONFIG),
    "--data-root", str(CDD11_ROOT),
    "--experiments-root", str(EXPERIMENTS_ROOT),
    "--nproc-per-node", "2",
], check=True)
TRUE_RUN = EXPERIMENTS_ROOT / "order_condition_true_ab_seed42_20ep"
FIXED_RUN = EXPERIMENTS_ROOT / "order_condition_fixed_ab_seed42_20ep"
TRUE_CHECKPOINT = TRUE_RUN / "best.pt"
FIXED_CHECKPOINT = FIXED_RUN / "best.pt"
assert TRUE_CHECKPOINT.is_file() and FIXED_CHECKPOINT.is_file()

In [ ]:
subprocess.run([
    "python", "-m", "hybrid_cot_nafnet.evaluate_order_condition_oracle",
    "--true-checkpoint", str(TRUE_CHECKPOINT),
    "--fixed-checkpoint", str(FIXED_CHECKPOINT),
    "--reference-json", str(REFERENCE),
    "--data-root", str(CDD11_ROOT),
    "--output-dir", str(EVALUATION_DIR),
    "--realizations", "3",
    "--bootstrap-samples", "5000",
], check=True)

In [ ]:
import pandas as pd
from IPython.display import display

decision = json.loads((EVALUATION_DIR / "decision.json").read_text())
contrasts = json.loads((EVALUATION_DIR / "contrasts.json").read_text())
print(json.dumps(decision, indent=2))
print(json.dumps(contrasts, indent=2))
display(pd.read_csv(EVALUATION_DIR / "per_order_all.csv"))

In [ ]:
import zipfile
from IPython.display import FileLink

required = ["decision.json", "contrasts.json", "per_order_all.csv", "per_sample_all.csv", "protocol.json"]
for name in required:
    assert (EVALUATION_DIR / name).is_file(), f"Missing evaluation artifact: {name}"
output_zip = Path("/kaggle/working/order_condition_oracle_results.zip")
with zipfile.ZipFile(output_zip, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(EVALUATION_DIR.rglob("*")):
        if path.is_file():
            archive.write(path, Path("evaluation") / path.relative_to(EVALUATION_DIR))
    archive.write(CONFIG, Path("protocol") / CONFIG.name)
    archive.write(REFERENCE, Path("protocol") / REFERENCE.name)
    archive.write(Path("docs/degradation_order_control_plan.md"), Path("protocol/degradation_order_control_plan.md"))
    archive.write(Path("/kaggle/working/order_condition_input_audit.json"), Path("protocol/order_condition_input_audit.json"))
    archive.write(Path("/kaggle/working/order_condition_data_gate.json"), Path("protocol/order_condition_data_gate.json"))
    for run_dir in (TRUE_RUN, FIXED_RUN):
        for name in ("run_summary.json", "run_config.json", "dataset_manifest.json", "runtime_resolution.json", "pretrained_report.json", "git_info.json", "train_log.csv", "memory_log.csv"):
            path = run_dir / name
            if path.is_file():
                archive.write(path, Path("training") / run_dir.name / name)
print("Download this ZIP and place it under results/:")
display(FileLink(str(output_zip)))